# Day 1 — Feel the tokens: cost, context, thinking and non-determinism (Gemini)
**Theory:** [day-01-what-is-an-agent.md](../theory/day-01-what-is-an-agent.md) · ⏱ ~45 min · Level: easy

**You will:**
1. Make your first Gemini call and read `usage_metadata` (tokens = money, or free-tier quota)
2. Count tokens *before* sending a request
3. See how a big system instruction changes every call
4. See **thinking tokens** and how the thinking level changes them
5. Plot why agent cost grows **faster than linearly** with steps
6. See that the same prompt gives different answers (why agents need evals)

> 🆓 **Free tier:** quotas are per minute and per day. The client retries automatically on `429`
> (rate limited), so a cell may pause for a few seconds. That's normal.

In [ ]:
# ⚙️ Setup: works in GitHub Codespaces, locally, in Colab and on Kaggle. Run this cell first.
# Please don't let an AI "auto-fix" rewrite the imports below: the course packages are `course_utils` and `shopease`.
import os, sys, pathlib, subprocess
IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:  # clone (or update) the course repo, install it, read the key from notebook secrets
    BRANCH = os.getenv("COURSE_BRANCH", "main")
    REPO_URL = os.getenv("COURSE_REPO_URL", "https://github.com/Rahulverma5/agentic-ai-course")
    REPO = pathlib.Path(os.getenv("COURSE_BASE_DIR", "/content" if IN_COLAB else "/kaggle/working")) / "agentic-ai-course"
    if not (REPO / ".git").exists():
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    else:  # an older clone from a previous session: bring it up to date
        subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "reset", "-q", "--hard", f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt"), "-e", str(REPO)],
                   check=True)
    try:
        if IN_COLAB:
            from google.colab import userdata
            os.environ.setdefault("GEMINI_API_KEY", userdata.get("GEMINI_API_KEY"))
        else:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("GEMINI_API_KEY", UserSecretsClient().get_secret("GEMINI_API_KEY"))
    except Exception as e:
        print("Add GEMINI_API_KEY to the notebook secrets (🔑 icon in Colab, Add-ons → Secrets in Kaggle):", e)
else:  # Codespaces / local: use the checked-out repo
    REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(REPO), str(REPO / "projects" / "shopease-support-agent")]
os.chdir(REPO)  # every path in this notebook is relative to the repo root
try:
    from google.genai import types  # Gemini SDK types: Content, Part, GenerateContentConfig, ...
    from course_utils import check_setup, client, MODEL, CHEAP_MODEL, ask, tracker, text_of, thinking, user
    from shopease.data import setup_data
except ImportError as e:
    raise ImportError(
        f"{e}\n\nThe course code at {REPO} is missing or outdated. Fix it like this:\n"
        f"  • Colab/Kaggle: run  !rm -rf {REPO}  then Runtime → Restart session, and run this cell again.\n"
        "  • Codespaces/local: run  git pull && pip install -e .  in a terminal, then restart the kernel.") from None
setup_data(quiet=True)  # creates the ShopEase sample data if it's missing
check_setup()                                   # checks your key and picks models your key can use
from course_utils.llm import MODEL, CHEAP_MODEL  # re-read: check_setup() may have switched them

## 1. Your first call: read the receipt

In [ ]:
r = client.models.generate_content(model=CHEAP_MODEL, contents="In 2 sentences: what is an AI agent?")
tracker.add(CHEAP_MODEL, r.usage_metadata, label="first call")
u = r.usage_metadata
print(text_of(r))
print(f"\nprompt tokens={u.prompt_token_count} · output tokens={u.candidates_token_count} · "
      f"thinking tokens={u.thoughts_token_count or 0}")
print("finish_reason:", r.candidates[0].finish_reason)

## 2. Count tokens before you send
`count_tokens` doesn't generate anything and tells you exactly how big a request is.

In [ ]:
policy_text = "\n".join(p.read_text() for p in sorted((REPO / "projects/shopease-support-agent/data/policies").glob("*.md")))
for name, text in [("short", "What is an AI agent?"), ("long", policy_text * 5 + "\n\nWhat is the return window?")]:
    n = client.models.count_tokens(model=MODEL, contents=text).total_tokens
    print(f"{name:>5}: {n:>6,} tokens  (~{len(text.split()):,} words)")

## 3. A big system instruction is sent on *every* call
Agents resend the system instruction on every loop step. Compare a tiny and a large one.

In [ ]:
big_system = "You are ShopEase support. Policies:\n" + policy_text * 8
for name, system in [("tiny system", "You are ShopEase support."), ("big system", big_system)]:
    r = client.models.generate_content(model=CHEAP_MODEL, contents="Say hi in 5 words.",
                                       config=types.GenerateContentConfig(system_instruction=system, max_output_tokens=100))
    cost = tracker.add(CHEAP_MODEL, r.usage_metadata, label=name)
    print(f"{name:<12} prompt={r.usage_metadata.prompt_token_count:>6,}  output={r.usage_metadata.candidates_token_count}"
          f"  ≈${cost:.5f}")

## 4. Thinking tokens
Modern Gemini models *think* before answering. Thinking tokens are billed like output tokens.
`thinking_level` (minimal / low / medium / high) is the knob, like the "effort" idea in the theory notes.

In [ ]:
question = "A train leaves at 9:40 and arrives at 13:05. How long is the trip? Answer in one line."
for level in ("minimal", "high"):
    try:
        r = client.models.generate_content(model=MODEL, contents=question,
                                           config=types.GenerateContentConfig(thinking_config=thinking(level)))
        tracker.add(MODEL, r.usage_metadata, label=f"thinking {level}")
        print(f"{level:<8} thinking tokens={r.usage_metadata.thoughts_token_count or 0:>5} → {text_of(r).strip()}")
    except Exception as e:  # some models don't support every level
        print(f"{level:<8} not supported by {MODEL}: {e}")

## 5. Why agent cost grows ~quadratically
At step *k* the prompt holds everything from steps 1..k-1. Total input = n·P₀ + a·n(n−1)/2.
(Prices: approximate paid Gemini Flash rates per 1M tokens. Day 5 shows how caching flattens this curve.)

In [ ]:
import matplotlib.pyplot as plt

def agent_cost(n_steps, p0=3000, add_per_step=800, out_per_step=300, in_price=1.50, out_price=7.50):
    inp = n_steps * p0 + add_per_step * n_steps * (n_steps - 1) / 2
    return (inp * in_price + n_steps * out_per_step * out_price) / 1e6

steps = list(range(1, 41))
plt.figure(figsize=(6, 3.5))
plt.plot(steps, [agent_cost(n) for n in steps], label="actual (history grows)")
plt.plot(steps, [agent_cost(1) * n for n in steps], "--", label="if it were linear")
plt.xlabel("agent steps"); plt.ylabel("$ per run (paid tier)"); plt.legend(); plt.title("Agent cost vs steps"); plt.show()
for n in (5, 10, 30):
    print(f"{n:>2} steps: ${agent_cost(n):.4f}")

## 6. Same prompt, different answers
LLMs sample. That's why you can't test an agent with one try. You need evals (Day 11).

In [ ]:
answers = [ask("Give ONE creative name for an AI support agent for an online store. Only the name.") for _ in range(5)]
for a in answers:
    print("-", a.strip())
print("\nunique answers:", len(set(a.strip() for a in answers)), "/ 5")

## 7. Reliability math: why recovery matters
A chain of *n* steps that each succeed with probability *p* succeeds with *pⁿ*.

In [ ]:
for p in (0.95, 0.99):
    print(f"p={p}: " + "  ".join(f"{n} steps → {p ** n:.2f}" for n in (5, 10, 30)))

## ✏️ Exercises
1. Change `add_per_step` in `agent_cost` to 3000 (big tool outputs). How much does a 30-step run cost now?
2. Compute: per-step accuracy 0.97, 20 steps, no recovery. What's the success rate?
3. Write one task where you'd use a **workflow** and one where you'd use an **agent**. Use the 4-point checklist from the theory note.
4. Open [AI Studio → Rate limits](https://aistudio.google.com/) and note your free-tier RPM/RPD for both models.

## 🧾 What did this notebook cost?
Every call above was recorded by `tracker` (estimated at paid prices; the free tier costs $0). Make a habit of checking tokens and cost after every experiment.

In [ ]:
print(tracker.summary())
tracker.to_dataframe().tail(10) if tracker.calls else None